<a href="https://colab.research.google.com/github/chetanbheem12-pande/gcolab/blob/main/ComfyUI%20Colab%20Upgraded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ ComfyUI: Professional Edition

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

### 🖥️ **High-Performance AI Environment**
This notebook provides a stable, persistent, and optimized environment for running ComfyUI on Google Colab.

**Key Features:**
- **Hybrid Storage:** Executes code on the local VM for speed, but persists data (Models, Output, Nodes) to Google Drive.
- **Modern Downloader:** Python-native, high-speed downloader with visual progress bars.
- **Memory Safety:** Prevents OOM (Out of Memory) errors via selectable GPU profiles.

In [4]:
#@title 1. System Initialization
#@markdown **Run this cell first.** <br>
#@markdown This mounts Drive and sets up your environment.

import os
import shutil
import subprocess
from google.colab import drive

# --- Configuration ---
MOUNT_DRIVE = True
UPDATE_COMFY_UI = True #@param {type:"boolean"}
INSTALL_COMFYUI_MANAGER = True #@param {type:"boolean"}

LOCAL_WORKSPACE = "/content/ComfyUI"
DRIVE_WORKSPACE = "/content/drive/MyDrive/ComfyUI"

# 1. Mount Google Drive
if MOUNT_DRIVE:
    print("💾 Mounting Google Drive...")
    drive.mount('/content/drive')

# 2. Setup ComfyUI Core
if not os.path.exists(LOCAL_WORKSPACE):
    print("📦 Cloning ComfyUI repository...")
    !git clone https://github.com/comfyanonymous/ComfyUI {LOCAL_WORKSPACE}
else:
    if UPDATE_COMFY_UI:
        print("🔄 Checking for ComfyUI updates...")
        !cd {LOCAL_WORKSPACE} && git pull

# 3. Configure Storage Symlinks
# Redirects heavy folders to Drive for persistence, keeps code local for speed.
directories_to_sync = [
    "models",
    "custom_nodes",
    "output",
    "input",
    "user"
]

print("🔗 Configuring persistent storage...")
for d in directories_to_sync:
    local_path = os.path.join(LOCAL_WORKSPACE, d)
    drive_path = os.path.join(DRIVE_WORKSPACE, d)

    # Ensure directory exists on Drive
    if not os.path.exists(drive_path):
        os.makedirs(drive_path, exist_ok=True)

    # Remove local directory to replace with symlink
    if os.path.exists(local_path) and not os.path.islink(local_path):
        shutil.rmtree(local_path)

    # Create Symlink
    if not os.path.exists(local_path):
        os.symlink(drive_path, local_path)

# 4. Install ComfyUI Manager (Optional)
if INSTALL_COMFYUI_MANAGER:
    manager_path = os.path.join(LOCAL_WORKSPACE, "custom_nodes", "ComfyUI-Manager")
    if not os.path.exists(manager_path):
        print("📦 Installing ComfyUI Manager...")
        !git clone https://github.com/ltdrdata/ComfyUI-Manager.git {manager_path}
    else:
        print("✅ ComfyUI Manager is installed.")
        !cd {manager_path} && git pull

# 5. Install Python Dependencies
print("🛠️ Installing Python environment requirements...")
!cd {LOCAL_WORKSPACE} && pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121
!pip install insightface onnxruntime-gpu

print("\n✅ [SYSTEM READY] Initialization complete.")

💾 Mounting Google Drive...
Mounted at /content/drive
📦 Cloning ComfyUI repository...
Cloning into '/content/ComfyUI'...
remote: Enumerating objects: 35135, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 35135 (delta 129), reused 85 (delta 85), pack-reused 34959 (from 3)
Receiving objects: 100% (35135/35135), 80.53 MiB | 19.29 MiB/s, done.
Resolving deltas: 100% (23788/23788), done.
🔗 Configuring persistent storage...
✅ ComfyUI Manager is installed.
Already up to date.
🛠️ Installing Python environment requirements...
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 136.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 MB 9.8 MB/s eta 0:00:00
   ━━━━━━

In [12]:
#@title 2. Model & Node Downloader
#@markdown Paste download links below. This supports **Checkpoints, Diffusion Models (Flux/UNET), Text Encoders (CLIP/T5), CLIP Vision, VAEs, LoRAs, and ControlNets**.

import os
import requests
from urllib.parse import urlparse, unquote
from tqdm.auto import tqdm

WORKSPACE = "/content/ComfyUI"

# --- Input Resources ---
CHECKPOINT_URLS = "" #@param {type:"string"}
UNET_DIFFUSION_URLS = "https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/ltx-2.3-22b-dev-Q4_K_M.gguf" #@param {type:"string"}
TEXT_ENCODER_URLS = "" #@param {type:"string"}
CLIP_VISION_URLS = "" #@param {type:"string"}
VAE_URLS = "" #@param {type:"string"}
LORA_URLS = "" #@param {type:"string"}
CONTROLNET_URLS = "" #@param {type:"string"}
UPSCALE_MODELS_URLS = "" #@param {type:"string"}
EMBEDDING_URLS = "" #@param {type:"string"}
CUSTOM_NODE_URLS = "" #@param {type:"string"}

# --- Downloader Logic ---
DIRS = {
    "checkpoints":    os.path.join(WORKSPACE, "models/checkpoints"),
    "unet":           os.path.join(WORKSPACE, "models/unet"),
    "clip":           os.path.join(WORKSPACE, "models/clip"),
    "clip_vision":    os.path.join(WORKSPACE, "models/clip_vision"),
    "vae":            os.path.join(WORKSPACE, "models/vae"),
    "loras":          os.path.join(WORKSPACE, "models/loras"),
    "controlnet":     os.path.join(WORKSPACE, "models/controlnet"),
    "upscale_models": os.path.join(WORKSPACE, "models/upscale_models"),
    "embeddings":     os.path.join(WORKSPACE, "models/embeddings"),
    "custom_nodes":   os.path.join(WORKSPACE, "custom_nodes")
}

def get_filename(url, response):
    """Smartly determines filename from Content-Disposition or URL."""
    if "Content-Disposition" in response.headers:
        import re
        fname = re.findall('filename="?([^"]+)"?', response.headers["Content-Disposition"])
        if fname: return fname[0]
    return unquote(os.path.basename(urlparse(url).path))

def download_file(url, target_dir):
    try:
        # Stream the download to get headers first
        response = requests.get(url, stream=True, allow_redirects=True)
        response.raise_for_status()

        filename = get_filename(url, response)
        file_path = os.path.join(target_dir, filename)
        total_size = int(response.headers.get('content-length', 0))

        if os.path.exists(file_path):
            print(f"   ⏩ Skipping (Exists): {filename}")
            return

        # Modern Progress Bar Log
        print(f"   📥 Downloading: {filename}")

        # The Progress Bar (Auto-Stretching)
        with tqdm(
            total=total_size,
            unit='B',
            unit_scale=True,
            unit_divisor=1024,
            desc="      🚀 Progress",
            dynamic_ncols=True
        ) as bar:
            with open(file_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=1024*1024): # 1MB chunks
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))

        print("      ✅ Download Complete\n")

    except Exception as e:
        print(f"   ❌ Failed to download: {url}")
        print(f"      Error: {e}\n")

def process_downloads(urls_str, target_dir, is_node=False):
    if not urls_str.strip(): return

    url_list = [u.strip() for u in urls_str.replace(',', '\n').split('\n') if u.strip()]
    if not os.path.exists(target_dir): os.makedirs(target_dir, exist_ok=True)

    print(f"📂 Category: {os.path.basename(target_dir)}")

    for url in url_list:
        if is_node:
            node_name = url.split('/')[-1].replace('.git', '')
            node_path = os.path.join(target_dir, node_name)
            if not os.path.exists(node_path):
                print(f"   ⬇️ Cloning Node: {node_name}...")
                !git clone {url} {node_path}
                # Auto-install requirements
                req = os.path.join(node_path, "requirements.txt")
                if os.path.exists(req):
                    print(f"      📦 Installing requirements...")
                    !pip install -r "{req}"
                print("      ✅ Installed\n")
            else:
                print(f"   ⏩ Node exists: {node_name}\n")
        else:
            download_file(url, target_dir)

# --- Execution ---
process_downloads(CHECKPOINT_URLS,     DIRS["checkpoints"])
process_downloads(UNET_DIFFUSION_URLS, DIRS["unet"])
process_downloads(TEXT_ENCODER_URLS,   DIRS["clip"])
process_downloads(CLIP_VISION_URLS,    DIRS["clip_vision"])
process_downloads(VAE_URLS,            DIRS["vae"])
process_downloads(LORA_URLS,           DIRS["loras"])
process_downloads(CONTROLNET_URLS,     DIRS["controlnet"])
process_downloads(UPSCALE_MODELS_URLS, DIRS["upscale_models"])
process_downloads(EMBEDDING_URLS,      DIRS["embeddings"])
process_downloads(CUSTOM_NODE_URLS,    DIRS["custom_nodes"], is_node=True)

print("🎉 All tasks finished.")

📂 Category: unet
   📥 Downloading: ltx-2.3-22b-dev-Q4_K_M.gguf


      🚀 Progress:   0%|          | 0.00/13.3G [00:00<?, ?B/s]

      ✅ Download Complete

🎉 All tasks finished.


In [2]:
import subprocess, os

# Fix 1: Install gguf permanently for this session
subprocess.run(["pip", "install", "gguf", "-q"], check=True)
print("✅ gguf installed")

# Fix 2: Install ngrok (replaces broken Cloudflare tunnel)
subprocess.run(["pip", "install", "pyngrok", "-q"], check=True)
print("✅ pyngrok installed")

# Verify gguf
import gguf

✅ gguf installed
✅ pyngrok installed


In [15]:
import subprocess, os

diff_path = "/content/drive/MyDrive/ComfyUI/models/diffusion_models"
dest = f"{diff_path}/ltx-video-2b-v0.9.1-Q4_K_S.gguf"

# Delete the corrupt empty file
if os.path.exists(dest):
    os.remove(dest)
    print("🗑️ Deleted empty file")

# Correct working URL
url = "https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/ltx-2.3-22b-dev-Q4_K_M.gguf"
fname = "ltx-2.3-22b-dev-Q4_K_M.gguf"
dest = f"{diff_path}/{fname}"

print(f"⬇️ Downloading {fname}...")
result = subprocess.run([
    "wget", "--show-progress",
    "--retry-connrefused", "--tries=3",
    "-O", dest, url
])

if os.path.exists(dest):
    size = os.path.getsize(dest)/1e9
    if size > 1:
        print(f"✅ Success: {fname} — {size:.1f}GB")
    else:
        print(f"❌ Still empty ({size:.2f}GB) — URL may be wrong")
else:
    print("❌ File not created")

🗑️ Deleted empty file
⬇️ Downloading ltx-2.3-22b-dev-Q4_K_M.gguf...
✅ Success: ltx-2.3-22b-dev-Q4_K_M.gguf — 14.3GB


In [16]:
import os

# Tell ComfyUI to look at Drive models directly — no symlinks needed
config = """
comfyui:
    base_path: /content/drive/MyDrive/ComfyUI/
    checkpoints: models/checkpoints/
    diffusion_models: models/diffusion_models/
    vae: models/vae/
    loras: models/loras/
    text_encoders: models/text_encoders/
    upscale_models: models/upscale_models/
    clip: models/text_encoders/
"""

config_path = "/content/ComfyUI/extra_model_paths.yaml"
with open(config_path, "w") as f:
    f.write(config)

print(f"✅ Config written to {config_path}")

# Verify your Drive models are there
base = "/content/drive/MyDrive/ComfyUI/models"
print("\n── Models on Drive ──")
for folder in ["checkpoints", "diffusion_models", "vae",
               "loras", "text_encoders", "upscale_models"]:
    path = f"{base}/{folder}"
    if os.path.exists(path) and not os.path.islink(path):
        files = [f for f in os.listdir(path) if not f.startswith('.')]
        for f in files:
            size = os.path.getsize(f"{path}/{f}")/1e9
            print(f"✅ {folder}/{f} — {size:.1f}GB")
    elif os.path.islink(path):
        print(f"⚠️  {folder}/ — is a symlink (points to {os.readlink(path)})")
    else:
        print(f"❌ {folder}/ — missing")

print("\n✅ Now re-run Cell 4 — ComfyUI will find all Drive models automatically!")

✅ Config written to /content/ComfyUI/extra_model_paths.yaml

── Models on Drive ──
✅ diffusion_models/ltx-2.3-22b-dev-Q4_K_M.gguf — 14.3GB
✅ vae/hunyuan_video_vae_bf16.safetensors — 0.5GB
✅ vae/ltx-2.3-22b-dev_audio_vae.safetensors — 0.0GB
✅ vae/ltx-2.3-22b-dev_video_vae.safetensors — 1.5GB
✅ vae/ltx-2.3-2b-dev_video_vae.safetensors — 0.0GB
✅ loras/ltx-2-19b-lora-camera-control-dolly-left.safetensors — 0.3GB
✅ loras/ltx-2.3-22b-distilled-lora-384.safetensors — 0.0GB
✅ text_encoders/gemma_3_12B_it_fp4_mixed.safetensors — 9.4GB
✅ upscale_models/ltx-2.3-spatial-upscaler-x2-1.0.safetensors — 0.0GB

✅ Now re-run Cell 4 — ComfyUI will find all Drive models automatically!


In [2]:
# @title 3. Session Anti-Disconnect
%%html
<b>🔊 Keep-Alive Audio</b><br>
<i>Running this silent audio loop prevents the browser tab from sleeping.</i><br>
<audio src="https://raw.githubusercontent.com/anars/blank-audio/master/10-minutes-of-silence.mp3" autoplay loop controls style="width: 300px;" />

In [ ]:
#@title 4. Start ComfyUI Session (ngrok version)
import subprocess, threading, time, socket, os
from pyngrok import ngrok, conf

# ── Config ──
MEMORY_PROFILE = "Standard (Auto-Detect)" #@param ["Standard (Auto-Detect)", "Low VRAM (T4 GPU / Heavy Models)", "High VRAM (A100 GPU Only)"]
LIVE_GENERATION_PREVIEWS = True #@param {type:"boolean"}
NGROK_TOKEN = "2oQZo3Kd6DABooV3WhsfZSMRx8O_469RJwW9a7Lpi3dZGUinR" #@param {type:"string"}

# ── Args ──
ARGS = ""
if "Low VRAM" in MEMORY_PROFILE:
    ARGS += " --lowvram"
elif "High VRAM" in MEMORY_PROFILE:
    ARGS += " --highvram"
if LIVE_GENERATION_PREVIEWS:
    ARGS += " --preview-method auto"

WORKSPACE = "/content/ComfyUI"
PORT = 8188

# ── Install gguf every session (required for GGUF nodes) ──
print("🔧 Installing gguf module...")
subprocess.run(["pip", "install", "gguf", "-q"])
print("✅ gguf ready")

# ── Start ngrok tunnel ──
def start_ngrok():
    # Wait for ComfyUI to be ready
    while True:
        time.sleep(1)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', PORT))
        sock.close()
        if result == 0:
            break

    print("\n🟢 ComfyUI is Online. Generating Access Link...\n")

    if NGROK_TOKEN:
        ngrok.set_auth_token(NGROK_TOKEN)

    tunnel = ngrok.connect(PORT, "http")
    url = tunnel.public_url
    print("\n" + "="*60)
    print(f"🔗 ACCESS LINK: {url}")
    print("="*60)
    print("\n✅ Click the link above to open ComfyUI")
    print("⚠️  If link shows error, wait 30 seconds and refresh")

threading.Thread(target=start_ngrok, daemon=True).start()

# ── Launch ComfyUI ──
if os.path.exists(os.path.join(WORKSPACE, "main.py")):
    %cd {WORKSPACE}
    print(f"🚀 Launching ComfyUI [{MEMORY_PROFILE}]...")
    os.system(f"python main.py --dont-print-server --listen 127.0.0.1 --port {PORT} {ARGS}")
else:
    print("❌ System files missing — re-run Cell 1")

🔧 Installing gguf module...
✅ gguf ready
/content/ComfyUI
🚀 Launching ComfyUI [Standard (Auto-Detect)]...

🟢 ComfyUI is Online. Generating Access Link...


🔗 ACCESS LINK: https://4d8b-34-158-60-24.ngrok-free.app

✅ Click the link above to open ComfyUI
⚠️  If link shows error, wait 30 seconds and refresh
